In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')

    GIT_BRANCH = 'refactor-continuo'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

In [ ]:
import json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

dataset_parquet = Path(f"{DATA_DIR}/processed/dataset/dataset.parquet")
legend_json = Path(f"{DATA_DIR}/processed/legend.json")

dataset = pd.read_parquet(dataset_parquet)

# Selezione dinamica delle feature predittive (esclude identificativi e label)
non_feature_cols = ["ID_Campo", "Ground_Truth", "Crop_Name"]
feature_cols = [c for c in dataset.columns if c not in non_feature_cols]

X = dataset[feature_cols]
y = dataset["Ground_Truth"].astype(int)

print(f"✅ Dataset caricato: {X.shape[0]} campioni e {X.shape[1]} feature predittive.")

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    # Nomi leggibili delle classi dal file di legenda
    target_names = None
    if legend_json.exists():
        with open(legend_json, "r", encoding="utf-8") as f:
            legend = json.load(f)
        unique_classes = sorted(y_test.unique())
        target_names = [f"{legend.get(str(c), c)} ({c})" for c in unique_classes]

    print("🏆 Classification Report 🏆")
    print("-" * 50)
    print(classification_report(y_test, prediction, target_names=target_names, zero_division=0, digits=3))
except Exception as e:
    print(f"Errore durante l'addestramento: {e}")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Calcolo della matrice di confusione
classes = sorted(y_test.unique())
cm = confusion_matrix(y_test, prediction, labels=classes)

# Normalizzazione per riga (Recall percentuale per ciascuna coltura)
with np.errstate(all='ignore'):
    cm_norm = np.nan_to_num(cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100)

# Etichette con nome della coltura e codice
legend_json = Path(f"{DATA_DIR}/processed/legend.json")
class_labels = [str(c) for c in classes]
if legend_json.exists():
    with open(legend_json, "r", encoding="utf-8") as f:
        legend = json.load(f)
    class_labels = [f"{legend.get(str(c), c)} ({c})" for c in classes]

plt.figure(figsize=(14, 11))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
    cbar_kws={"label": "Recall per Classe (%)"},
    linewidths=0.5,
    linecolor="#e0e0e0"
)

plt.title("Matrice di Confusione Normalizzata (Recall % per Coltura)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Classe Predetta", fontsize=12, labelpad=10)
plt.ylabel("Classe Reale (Ground Truth)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()